# 02 · Benchmark: questions and reference answers

The first version of the natural-language benchmark (point 5 of Myles's week-2 instructions): 15 simple spatial
questions. **Every question is answered by one call of one of the three tested operations**
(`src/space_mel/operations.py`) and the answer is read directly from its output, never computed by an LLM.

The questions come in **tasks**: the same task asked in two different wordings, with exactly the same operation and
inputs, so later we can check whether the LLM understands the meaning and not just the words.

| Operation | What it returns | Question form |
|---|---|---|
| `spots_within_radius(spots, spot, r)` | the spots within *r* µm of one spot | which spots are near spot X? |
| `distance_to_region(spots, region)` | for every spot, the distance (µm) to the nearest region spot | how far is spot X from region Y? |
| `neighbourhood_fraction(spots, category, r)` | for every spot, neighbours / in the category / fraction | what share of spot X's neighbours are in Y? |

**Status:** *answered* = reference answer computed below; *awaiting annotation* = needs real nevus/melanoma labels.

**Data:** a drawn **toy grid** (answers can be checked by eye) and patient **A24** (filtered spots), with two sets
defined by **one gene each** (expression only, no biology claim): **PRAME-high** = PRAME > 1.5 and **CXCL9-high** =
CXCL9 > 2. "Touching" = within **150 µm** (see notebook 01).

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "02_spatial_operations"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.operations import distance_to_region, neighbourhood_fraction, spots_table, spots_within_radius
from space_mel.spatial import grid_to_um

FIGS = STAGE / "results"
OUT = STAGE / "benchmark"
OUT.mkdir(exist_ok=True)
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#d9d9d6", "#52514e"
TOUCHING_UM = 150
X = "AACTGCCGCTACATCT-1"   # A24 spot used in the neighbour and fraction questions
Y = "GACGCTCGAGGAGTGC-1"   # A24 spot used in the distance questions
answers = {}               # task -> reference answer, filled in below

## The 15 questions

In [ ]:
W100, W150 = "radius 100 µm", "radius 150 µm (touching)"
SETS = "PRAME-high = PRAME > 1.5; CXCL9-high = CXCL9 > 2"
Q = [
    # task, question, data, operation, inputs, parameters, assumptions, status
    ("T1", "Which spots are within 100 µm of spot r0c0?", "toy grid", "spots_within_radius", "spot r0c0", W100, "none", "answered"),
    ("T1", "List the neighbours of spot r0c0 that are at most 100 µm away.", "toy grid", "spots_within_radius", "spot r0c0", W100, "none", "answered"),
    ("T2", f"Which spots touch spot {X}?", "A24", "spots_within_radius", f"spot {X}", W150, "touching = within 150 µm", "answered"),
    ("T2", f"Which spots are directly next to spot {X}?", "A24", "spots_within_radius", f"spot {X}", W150, "next to = within 150 µm", "answered"),
    ("T3", "How far is spot r2c4 from region R?", "toy grid", "distance_to_region", "spot r2c4; region R", "µm", "none", "answered"),
    ("T3", "What is the distance between spot r2c4 and the closest spot of region R?", "toy grid", "distance_to_region", "spot r2c4; region R", "µm", "none", "answered"),
    ("T4", f"How far is spot {Y} from the nearest PRAME-high spot?", "A24", "distance_to_region", f"spot {Y}; region PRAME-high", "µm", SETS, "answered"),
    ("T4", f"What is the distance from spot {Y} to the PRAME-high spots?", "A24", "distance_to_region", f"spot {Y}; region PRAME-high", "µm", SETS, "answered"),
    ("T5", "What fraction of the neighbours of spot r0c0 (within 100 µm) are in category C?", "toy grid", "neighbourhood_fraction", "spot r0c0; category C", W100, "none", "answered"),
    ("T5", "Of the spots within 100 µm of r0c0, what share belongs to category C?", "toy grid", "neighbourhood_fraction", "spot r0c0; category C", W100, "none", "answered"),
    ("T6", f"What fraction of the spots touching spot {X} are CXCL9-high?", "A24", "neighbourhood_fraction", f"spot {X}; category CXCL9-high", W150, SETS, "answered"),
    ("T6", f"Among the direct neighbours of spot {X}, what proportion is CXCL9-high?", "A24", "neighbourhood_fraction", f"spot {X}; category CXCL9-high", W150, SETS, "answered"),
    ("T7", "Which spots are within 100 µm of spot 'alone'?", "toy grid", "spots_within_radius", "spot alone", W100, "none", "answered"),
    ("T8", "How far is spot s1 (section A22) from region {s2} (section A24)?", "two-section toy", "distance_to_region", "spot s1; region {s2}", "µm", "sections never connect", "answered"),
    ("T9", f"How far is spot {Y} from the nevus?", "A24", "distance_to_region", f"spot {Y}; region nevus", "µm", "needs a nevus label per spot", "awaiting annotation"),
]
questions = pd.DataFrame(Q, columns=["task", "question", "data", "operation", "inputs", "parameters", "assumptions", "status"])
questions.index = [f"B{i:02d}" for i in range(1, len(Q) + 1)]
questions.index.name = "id"
questions.groupby(["operation", "status"]).size().rename("questions").to_frame()

## The data

### Toy grid
A piece of the Visium grid plus one isolated spot. Region **R** and category **C** are the orange rings. Spot names
are `r<row>c<column>`.

In [ ]:
rows, cols = np.meshgrid(np.arange(-2, 3), np.arange(-4, 5), indexing="ij")
keep = (rows + cols) % 2 == 0                         # Visium uses every other (row, col) position
xy = grid_to_um(rows[keep], cols[keep])
toy = pd.DataFrame({"x_um": xy[:, 0], "y_um": xy[:, 1], "section": "toy"},
                   index=[f"r{r}c{c}" for r, c in zip(rows[keep], cols[keep])])
toy.loc["alone"] = [700.0, 0.0, "toy"]

R = pd.Series(toy.index.isin(["r-2c-4", "r-2c-2", "r-1c-3"]), index=toy.index)
C = pd.Series(toy.index.isin(["r0c2", "r1c1", "r-1c1", "r1c-1"]), index=toy.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, ring, name in zip(axes, [R, C], ["region R", "category C"]):
    ax.scatter(toy.x_um, toy.y_um, s=900, c=GREY, linewidths=0)
    ax.scatter(toy.x_um[ring], toy.y_um[ring], s=900, facecolors="none", edgecolors=ORANGE, linewidths=2.5)
    for b in toy.index:
        ax.annotate(b, (toy.x_um[b], toy.y_um[b]), ha="center", va="center", fontsize=7)
    ax.set_xlim(-480, 780); ax.set_ylim(250, -250)       # y downwards, room for the large markers
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"Toy grid: {name} (orange rings)", loc="left")
fig.savefig(FIGS / "benchmark_toy_grid.png", bbox_inches="tight")

### A24: two sets defined by one gene each, and the two example spots

In [ ]:
a24 = ad.read_h5ad(ROOT / "data/processed/GSE298774_A24.h5ad")   # filtered spots
spots = spots_table(a24)

def gene(name):
    return pd.Series(a24[:, name].X.toarray().ravel(), index=a24.obs_names)

prame_high = gene("PRAME") > 1.5
cxcl9_high = gene("CXCL9") > 2
print(f"A24: {len(spots)} spots | PRAME-high: {prame_high.sum()} | CXCL9-high: {cxcl9_high.sum()}")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
colors = np.select([prame_high & cxcl9_high, cxcl9_high, prame_high], [INK, ORANGE, BLUE], GREY)
ax.scatter(spots.x_um, spots.y_um, c=colors, s=6, linewidths=0)
for spot, label in [(X, "X"), (Y, "Y")]:
    ax.scatter(spots.at[spot, "x_um"], spots.at[spot, "y_um"], s=80, facecolors="none", edgecolors="black")
    ax.annotate(f"spot {label}", (spots.at[spot, "x_um"] + 60, spots.at[spot, "y_um"]), fontsize=9)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.set_title("A24: PRAME-high (blue), CXCL9-high (orange), both (dark grey)", loc="left")
fig.savefig(FIGS / "benchmark_A24_sets.png", bbox_inches="tight")

## Reference answers: one operation call per task

Both questions of a task get the same answer. Each block is the whole procedure.

**T1** (B01, B02): spots within 100 µm of r0c0 (toy)

In [ ]:
answers["T1"] = sorted(spots_within_radius(toy, "r0c0", 100))
answers["T1"]

**T2** (B03, B04): spots touching spot X (A24)

In [ ]:
answers["T2"] = sorted(spots_within_radius(spots, X, TOUCHING_UM))
answers["T2"]

**T3** (B05, B06): distance from r2c4 to region R (toy)

In [ ]:
answers["T3"] = distance_to_region(toy, R)["r2c4"]
answers["T3"]

**T4** (B07, B08): distance from spot Y to the nearest PRAME-high spot (A24)

In [ ]:
answers["T4"] = distance_to_region(spots, prame_high)[Y]
answers["T4"]

**T5** (B09, B10): share of r0c0's neighbours (within 100 µm) in category C (toy)

In [ ]:
answers["T5"] = neighbourhood_fraction(toy, C, 100).loc["r0c0"]
answers["T5"]

**T6** (B11, B12): share of the spots touching X that are CXCL9-high (A24)

In [ ]:
answers["T6"] = neighbourhood_fraction(spots, cxcl9_high, TOUCHING_UM).loc[X]
answers["T6"]

**T7** (B13): spots within 100 µm of the isolated spot (toy): an empty answer is the correct answer

In [ ]:
answers["T7"] = sorted(spots_within_radius(toy, "alone", 100))
answers["T7"]

**T8** (B14): two spots 10 µm apart but in different sections: no distance (NaN)

In [ ]:
two = pd.DataFrame({"x_um": [0.0, 10.0], "y_um": [0.0, 0.0], "section": ["A22", "A24"]}, index=["s1", "s2"])
answers["T8"] = distance_to_region(two, pd.Series([False, True], index=two.index))["s1"]
answers["T8"]

## The benchmark table

B15 (T9) has no answer: there is no nevus label. Saved as `02_spatial_operations/benchmark/benchmark.csv`.

In [ ]:
def as_text(value):
    if isinstance(value, list):
        return f"{len(value)} spots: " + ", ".join(value) if value else "none (empty list)"
    if isinstance(value, pd.Series):                   # a row of neighbourhood_fraction
        n, k, f = int(value.n_neighbours), int(value.n_in_category), value.fraction
        return f"{k}/{n} = {f:.2f}"
    return "not defined (NaN)" if np.isnan(value) else f"{value:.1f} µm"

questions["reference_answer"] = questions.task.map({t: as_text(v) for t, v in answers.items()}).fillna("-")
questions.to_csv(OUT / "benchmark.csv")
with pd.option_context("display.max_colwidth", 70):
    display(questions[["task", "question", "operation", "reference_answer"]])